In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import f1_score, make_scorer, confusion_matrix

DATA_DIR = Path(".")  # 이 Notebook이 들어 있는 problem_04 폴더
CHANNELS = ["force_x", "force_y", "force_z", "torque_x", "torque_y", "torque_z"]
train_series = pd.read_csv(DATA_DIR / "train_series.csv")
test_series = pd.read_csv(DATA_DIR / "test_series.csv")
labels = pd.read_csv(DATA_DIR / "train_labels.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")

def pack_sequences(frame, ordered_ids):
    if set(frame.sample_id) != set(ordered_ids):
        raise ValueError("ID mismatch")
    groups = {key: group.sort_values("step") for key, group in frame.groupby("sample_id")}
    arrays = []
    for key in ordered_ids:
        group = groups[key]
        if not np.array_equal(group.step.to_numpy(), np.arange(15)):
            raise ValueError(f"Invalid steps: {key}")
        values = group[CHANNELS].to_numpy(dtype=float)
        if not np.isfinite(values).all():
            raise ValueError(f"Missing/nonfinite values: {key}")
        arrays.append(values)
    return np.stack(arrays)

X_seq = pack_sequences(train_series, labels.sample_id)
X_test_seq = pack_sequences(test_series, sample.sample_id)
y = labels.label.to_numpy()
print("Train:", X_seq.shape, "Test:", X_test_seq.shape)
print("Label counts:", labels.label.value_counts().sort_index().to_dict())

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Train: (120, 15, 6) Test: (41, 15, 6)
Label counts: {0: 33, 1: 19, 2: 33, 3: 16, 4: 19}


In [2]:
# 채점 방식과 동일한 Macro-F1
LABELS = [0, 1, 2, 3, 4]
def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0)
scorer = make_scorer(macro_f1)
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

In [3]:
def stats_features(s):  # 36개: 채널별 mean, std, min, max, 끝-처음, 평균 |변화량|
    return np.hstack([s.mean(1), s.std(1), s.min(1), s.max(1),
                      s[:, -1] - s[:, 0], np.abs(np.diff(s, axis=1)).mean(1)])

def segment_means(s):  # 18개: step 0~4, 5~9, 10~14 구간 평균
    return np.hstack([s[:, a:a + 5].mean(1) for a in (0, 5, 10)])

def make_model():
    return ExtraTreesClassifier(n_estimators=500, class_weight="balanced", random_state=42)

def save_submission(pred):
    submission = pd.DataFrame({"sample_id": sample.sample_id, "label": pred.astype(int)})
    submission.to_csv(DATA_DIR / "submission.csv", index=False, encoding="utf-8-sig")
    print("Saved:", (DATA_DIR / "submission.csv").resolve())
    print("Pred label counts:", submission.label.value_counts().sort_index().to_dict())

In [4]:
# 안전 제출본: 통계36 + ExtraTrees
safe_model = make_model().fit(stats_features(X_seq), y)
save_submission(safe_model.predict(stats_features(X_test_seq)))

Saved: /home/akswosla/maicon/problem_04/submission.csv
Pred label counts: {0: 12, 1: 4, 2: 12, 3: 6, 4: 7}


In [5]:
# 후보 비교 (반복 계층 5-fold x 3)
candidates = {
    "base (stats36)": lambda s: stats_features(s),
    "A (stats36 + segment18)": lambda s: np.hstack([stats_features(s), segment_means(s)]),
}
cv_scores = {}
for name, feat in candidates.items():
    scores = cross_val_score(make_model(), feat(X_seq), y, cv=cv, scoring=scorer)
    cv_scores[name] = scores.mean()
    print(f"{name:26s} {scores.mean():.3f} ± {scores.std():.3f}")

# 채택 기준: base 대비 +0.02 이상일 때만 A 채택
chosen = "A (stats36 + segment18)" if cv_scores["A (stats36 + segment18)"] - cv_scores["base (stats36)"] >= 0.02 else "base (stats36)"
print("Chosen:", chosen)

oof = cross_val_predict(make_model(), candidates[chosen](X_seq), y, cv=StratifiedKFold(5, shuffle=True, random_state=42))
print("OOF Macro-F1:", round(macro_f1(y, oof), 3))
print("Confusion matrix (rows=true, columns=predicted):")
print(confusion_matrix(y, oof, labels=LABELS))

base (stats36)             0.763 ± 0.071


A (stats36 + segment18)    0.768 ± 0.085
Chosen: base (stats36)


OOF Macro-F1: 0.77
Confusion matrix (rows=true, columns=predicted):
[[31  1  1  0  0]
 [ 0  6 13  0  0]
 [ 2  3 28  0  0]
 [ 0  0  0 14  2]
 [ 0  0  2  1 16]]


In [6]:
# 클래스 1 확률 가중치 검증 (OOF 확률, 5-fold x 10회)
CLASS1_WEIGHT = 1.4

def predict_weighted(model, features):
    proba = model.predict_proba(features)
    proba[:, list(model.classes_).index(1)] *= CLASS1_WEIGHT
    return model.classes_[proba.argmax(1)]

X_chosen = candidates[chosen](X_seq)
weight_scores = {1.0: [], CLASS1_WEIGHT: []}
for rep in range(10):
    proba = np.zeros((len(y), 5))
    for fit_idx, valid_idx in StratifiedKFold(5, shuffle=True, random_state=rep).split(X_chosen, y):
        proba[valid_idx] = make_model().fit(X_chosen[fit_idx], y[fit_idx]).predict_proba(X_chosen[valid_idx])
    for w in weight_scores:
        weighted = proba.copy()
        weighted[:, 1] *= w
        weight_scores[w].append(macro_f1(y, weighted.argmax(1)))
for w, scores in weight_scores.items():
    print(f"class1 weight {w}: OOF Macro-F1 {np.mean(scores):.3f} ± {np.std(scores):.3f}")

class1 weight 1.0: OOF Macro-F1 0.725 ± 0.037
class1 weight 1.4: OOF Macro-F1 0.758 ± 0.031


In [7]:
# 최종 학습·예측·저장
feat = candidates[chosen]
final_model = make_model().fit(feat(X_seq), y)
test_pred = predict_weighted(final_model, feat(X_test_seq))
save_submission(test_pred)
missing = set(LABELS) - set(test_pred)
print("All 5 classes predicted" if not missing else f"WARNING: classes never predicted: {missing}")

Saved: /home/akswosla/maicon/problem_04/submission.csv
Pred label counts: {0: 11, 1: 8, 2: 9, 3: 6, 4: 7}
All 5 classes predicted


In [8]:
# 형식 검사일 뿐 정답 채점이 아닙니다.
check = pd.read_csv(DATA_DIR / "submission.csv")
assert list(check.columns) == ["sample_id", "label"]
assert len(check) == 41 and check.sample_id.is_unique
assert set(check.sample_id) == set(sample.sample_id)
assert pd.api.types.is_integer_dtype(check.label)
assert check.label.isin(range(5)).all()
print("CSV format OK.")

CSV format OK.
